In [1]:
# imports
import os, sys, time

# import the __functions.py (custom functions)
sys.path.append(os.getcwd()) # add code folder to system path
from __functions import *  # imports all custom functions

# local data directories
datadir = '/Users/mcc/Library/CloudStorage/Box-Box/MCC/data/'
projdir = os.path.dirname(os.getcwd())
print(projdir)

print("Complete!")

/Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation
Complete!


In [10]:
# --- Read in the incidents perimeter data
fp = os.path.join(projdir, 'data/spatial/mod/ics209plus_2014to2023_perimeters_tcc.gpkg')
incis = gpd.read_file(fp)
print(incis.columns)

# --- Keep only forested incidents
incis_ = incis.copy()
incis_ = incis_[
    (incis_['FUEL_MODEL'].str.contains('Timber', case=False)) |
    (incis_['tcc']>=10) # keep forested incidents
]
print(len(incis_))
print(len(incis_[
    (incis_['tcc']>=10)
]))
print(len(incis_[
    (incis_['FUEL_MODEL'].str.contains('Timber', case=False))
]))

Index(['CAUSE', 'COMPLEX', 'DISCOVERY_DATE', 'DISCOVERY_DOY',
       'EVACUATION_REPORTED', 'EXPECTED_CONTAINMENT_DATE', 'FATALITIES',
       'FATALITIES_PUBLIC', 'FATALITIES_RESPONDER', 'FINAL_ACRES',
       'FINAL_REPORT_DATE', 'FIRE_SIZE_CLASS', 'FIRE_YEAR', 'FOD_CONT_DATE',
       'FOD_CONT_DOY', 'FOD_CONT_TIME', 'FOD_DISCOVERY_DATE',
       'FOD_DISCOVERY_DOY', 'FOD_DISCOVERY_TIME', 'FOD_EXCL_CAT',
       'FOD_FIRE_SIZE', 'FOD_ID', 'FOD_INCIDENT_GROUP_NAME', 'FOD_LATITUDE',
       'FOD_LONGITUDE', 'FUEL_MODEL', 'INCIDENT_DESCRIPTION', 'INCIDENT_ID',
       'INCIDENT_ID_OLD', 'INCIDENT_NAME', 'INCIDENT_NUMBER',
       'INCTYP_ABBREVIATION', 'INC_IDENTIFIER', 'INC_MGMT_NUM_SITREPS',
       'INJURIES_TOTAL', 'IRWIN_ID', 'LL_CONFIDENCE', 'LL_UPDATE',
       'LOCAL_TIMEZONE', 'MTBS_FIRE_NAME', 'MTBS_ID',
       'NWCG_CAUSE_CLASSIFICATION', 'NWCG_GENERAL_CAUSE', 'PEAK_EVACUATIONS',
       'POO_CITY', 'POO_COUNTY', 'POO_LATITUDE', 'POO_LONGITUDE',
       'POO_SHORT_LOCATION_DESC', 'POO_S

In [14]:
print(len(incis_[~incis_['MTBS_ID'].isna()]))

2276


In [9]:
incis_['match_method'].value_counts()

match_method
MTBS          2224
NIFC_IRWIN    1551
WUMI_IRWIN     403
WUMI_FOD        81
Name: count, dtype: int64

In [2]:
# load the NSI extracted data
nsi_fire = os.path.join(projdir,"data/spatial/mod/nsi/nsi_structures_in_fires_2014to2023.gpkg")
nsi_fire = gpd.read_file(nsi_fire)
nsi_fire.columns

Index(['fd_id', 'bid', 'occtype', 'st_damcat', 'bldgtype', 'found_type',
       'cbfips', 'pop2amu65', 'pop2amo65', 'pop2pmu65', 'pop2pmo65', 'sqft',
       'num_story', 'ftprntid', 'ftprntsrc', 'students', 'found_ht',
       'val_struct', 'val_cont', 'val_vehic', 'source', 'med_yr_blt',
       'firmzone', 'o65disable', 'u65disable', 'x', 'y', 'ground_elv',
       'ground_elv_m', 'FIPS', 'index_right', 'INCIDENT_ID', 'geometry'],
      dtype='object')

In [3]:
# check on NAs and summary stats
# how many structures total?
print(f"Found {len(nsi_fire)} NSI structure in fire perimeters.")

Found 285625 NSI structure in fire perimeters.


In [4]:
# calculate the median value for residential and non-residential for each fire
df = nsi_fire.copy()
df['RES'] = df['st_damcat'].apply(lambda x: 'R' if x == 'RES' else 'NR')
# make sure there is valid data to play with
df = df[
    df["val_struct"].notnull() &
    (df["val_struct"] > 0)
]
# group by fire and building type (residential / non-residential), and calculate statistics
grouped = df.groupby(["INCIDENT_ID", "RES"], observed=True).agg(
    val_struct = ('val_struct', 'median'),
    n_struct = ('val_struct', 'count')
).reset_index()

# Pivot to wide format: one row per fire, one column per building type
res_val = grouped.pivot(index=["INCIDENT_ID"], columns="RES", values=["val_struct","n_struct"])
res_val.columns = [f"{var}_{res}" for var, res in res_val.columns]
res_val = res_val.reset_index()

# handle NaNs
res_val['n_struct_R'] = res_val['n_struct_R'].fillna(0)
res_val['n_struct_NR'] = res_val['n_struct_NR'].fillna(0)

# create a proportion column
res_val['n_struct_total'] = res_val['n_struct_R'] + res_val['n_struct_NR']
res_val['prop_R'] = res_val['n_struct_R'] / res_val['n_struct_total']

# round the USD values
res_val['val_struct_R'] = res_val['val_struct_R'].round(2).astype("float64")
res_val['val_struct_NR'] = res_val['val_struct_NR'].round(2).astype("float64")

# adjust to 2024 dollars
res_val['val_struct_NR24'] = res_val['val_struct_NR'] * 1.082755
res_val['val_struct_R24'] = res_val['val_struct_R'] * 1.082755

res_val.head()

,INCIDENT_ID,val_struct_NR,val_struct_R,n_struct_NR,n_struct_R,n_struct_total,prop_R,val_struct_NR24,val_struct_R24
0,2014_1077684_APPLEGATE,546439.74,216775.43,27.0,216.0,243.0,0.888889,5.916604e+05,234714.680710
1,2014_1126016_HIGHWAY 613 FIRE,1285209.50,152487.98,7.0,185.0,192.0,0.963542,1.391567e+06,165107.122785
2,2014_1138291_WEST RANGE FIRE,NaN,183383.15,0.0,11.0,11.0,1.000000,NaN,198559.022578
3,2014_1165446_MAGNOLIA SPRINGS,NaN,125111.97,0.0,14.0,14.0,1.000000,NaN,135465.611077
4,2014_1173585_ALAQUA,162952.00,262009.73,1.0,1.0,2.0,0.500000,1.764371e+05,283692.345206


In [5]:
# save out.
out_fp = os.path.join(projdir,"data/tabular/NSI_summary.csv")
res_val.to_csv(out_fp, index=False)
print(f"Saved to: {out_fp}")

Saved to: /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/tabular/NSI_summary.csv
